# 🏪 Dark Store AI — Inventory Forecasting Explorer

Interactive notebook for exploring demand patterns, running forecasts, and reviewing inventory alerts.

**Sections:**
1. Data Overview
2. ABC Analysis (Top SKUs by Volume)
3. Demand Patterns
4. Forecast Visualization
5. Inventory Alerts
6. V2 Ideas

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add project root to path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

from src.data_loader import load_orders, load_inventory, get_top_skus
from src.features import build_forecast_features
from src.forecaster import forecast_sku, forecast_all_top_skus
from src.inventory_alerts import compute_expiry_risk, compute_reorder_alerts, summarize_alerts

print('✅ Imports OK')
from matplotlib.patches import Patch

---
## 1. Data Overview

In [ ]:
orders_df = load_orders(ROOT / 'data' / 'sample_orders.csv')
inventory_df = load_inventory(ROOT / 'data' / 'sample_inventory.csv')

print(f'Orders shape   : {orders_df.shape}')
print(f'Inventory shape: {inventory_df.shape}')
print(f'Date range     : {orders_df["order_date"].min().date()} → {orders_df["order_date"].max().date()}')
print(f'Stores         : {sorted(orders_df["store_id"].unique())}')
print(f'Categories     : {sorted(orders_df["category"].unique())}')
print(f'Unique SKUs    : {orders_df["sku_id"].nunique()}')

In [ ]:
print('--- Orders sample ---')
display(orders_df.head(5))
print('\n--- Inventory sample ---')
display(inventory_df.head(5))

In [ ]:
# Daily total demand across all stores
daily_total = orders_df.groupby('order_date')['quantity'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(daily_total['order_date'], daily_total['quantity'], alpha=0.4, color='steelblue')
ax.plot(daily_total['order_date'], daily_total['quantity'], color='steelblue', linewidth=1)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.set_title('Total Daily Demand — All Stores & SKUs', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Units Sold')
plt.tight_layout()
plt.show()

---
## 2. ABC Analysis — Top SKUs by Volume

In [ ]:
top_skus = get_top_skus(orders_df, n=20)
display(top_skus)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart of top 20
ax1 = axes[0]
colors = ['#e74c3c' if i < 6 else '#f39c12' if i < 14 else '#2ecc71' for i in range(len(top_skus))]
bars = ax1.barh(top_skus['sku_name'][::-1], top_skus['total_quantity'][::-1], color=colors[::-1])
ax1.set_xlabel('Total Units Sold')
ax1.set_title('Top 20 SKUs by Volume (All Stores)', fontweight='bold')
ax1.tick_params(axis='y', labelsize=9)

# Cumulative % (Pareto)
ax2 = axes[1]
ax2.plot(range(1, len(top_skus) + 1), top_skus['cumulative_pct'], marker='o', color='steelblue')
ax2.axhline(80, linestyle='--', color='red', alpha=0.7, label='80% threshold')
ax2.set_xlabel('Number of SKUs')
ax2.set_ylabel('Cumulative % of Volume')
ax2.set_title('Pareto Chart — Cumulative Demand %', fontweight='bold')
ax2.legend()
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()
print(f"Top 6 SKUs account for {top_skus.iloc[5]['cumulative_pct']:.1f}% of all volume")

---
## 3. Demand Patterns

In [ ]:
STORE_ID = 'STORE_A'
top3_skus = top_skus.head(3)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, (_, sku_row) in zip(axes, top3_skus.iterrows()):
    features = build_forecast_features(orders_df, sku_row['sku_id'], STORE_ID)
    if features is None:
        continue
    # 7-day rolling average
    ax.fill_between(features['ds'], features['daily_demand'], alpha=0.3, color='steelblue')
    ax.plot(features['ds'], features['rolling_7d_avg'], color='tomato', linewidth=2, label='7-day avg')
    ax.set_title(f"{sku_row['sku_name']} — Daily Demand ({STORE_ID})", fontsize=11)
    ax.set_ylabel('Units')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.xlabel('Date')
plt.suptitle('Daily Demand Patterns — Top 3 SKUs', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Weekday vs weekend demand comparison for top SKU
top_sku = top_skus.iloc[0]
features = build_forecast_features(orders_df, top_sku['sku_id'], STORE_ID)

dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_avg = features.groupby('day_of_week')['daily_demand'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ['#e74c3c' if d >= 5 else '#3498db' for d in range(7)]
ax.bar(dow_labels, [dow_avg.get(i, 0) for i in range(7)], color=bar_colors)
ax.set_title(f"{top_sku['sku_name']} — Avg Demand by Day of Week", fontweight='bold')
ax.set_ylabel('Avg Units/Day')
ax.legend(handles=[Patch(color='#3498db', label='Weekday'), Patch(color='#e74c3c', label='Weekend')])
plt.tight_layout()
plt.show()

---
## 4. Forecast Visualization

In [ ]:
# Forecast the top SKU using LightGBM (faster than Prophet in notebook)
FORECAST_SKU = top_skus.iloc[0]['sku_id']
FORECAST_METHOD = 'lgbm'  # change to 'prophet' if installed
HORIZON = 14

print(f'Forecasting {top_skus.iloc[0]["sku_name"]} for {STORE_ID} using {FORECAST_METHOD}...')
forecast_df = forecast_sku(orders_df, FORECAST_SKU, STORE_ID, horizon=HORIZON, method=FORECAST_METHOD)
display(forecast_df)

In [ ]:
# Build historical context for the chart
hist = build_forecast_features(orders_df, FORECAST_SKU, STORE_ID)
last_30 = hist.tail(30)

fig, ax = plt.subplots(figsize=(14, 5))

# Historical actual
ax.plot(last_30['ds'], last_30['daily_demand'], color='steelblue', linewidth=1.5, label='Actual demand')
ax.fill_between(last_30['ds'], last_30['daily_demand'], alpha=0.15, color='steelblue')

# Forecast
ax.plot(forecast_df['ds'], forecast_df['yhat'], color='tomato', linewidth=2, linestyle='--', label=f'{FORECAST_METHOD} forecast')
ax.fill_between(forecast_df['ds'], forecast_df['yhat_lower'], forecast_df['yhat_upper'],
                alpha=0.25, color='tomato', label='80% CI')

# Dividing line
split_date = forecast_df['ds'].min()
ax.axvline(split_date, color='gray', linestyle=':', alpha=0.8)
ax.text(split_date, ax.get_ylim()[1] * 0.95, ' Forecast →', color='gray', fontsize=9)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.set_title(f"{top_skus.iloc[0]['sku_name']} — 30-Day History + {HORIZON}-Day Forecast ({STORE_ID})",
             fontweight='bold')
ax.set_ylabel('Units / Day')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Inventory Alerts

In [ ]:
# Generate forecasts for all top SKUs
print(f'Running {FORECAST_METHOD} forecasts for all top {len(top_skus)} SKUs...')
forecasts_dict = forecast_all_top_skus(orders_df, top_skus, STORE_ID, horizon=7, method=FORECAST_METHOD)

In [ ]:
# Expiry risk
store_inventory = inventory_df[inventory_df['store_id'] == STORE_ID].copy()
expiry_df = compute_expiry_risk(store_inventory, forecasts_dict)

print('--- Expiry Risk Summary ---')
print(expiry_df['expiry_risk'].value_counts())
print()

# Show high + medium risk items
at_risk = expiry_df[expiry_df['expiry_risk'].isin(['HIGH', 'MEDIUM'])].sort_values(
    ['expiry_risk', 'sell_through_probability'])
display(at_risk[['sku_name', 'current_stock', 'expiry_days', 'forecasted_demand_before_expiry',
                  'sell_through_probability', 'expiry_risk']].reset_index(drop=True))

In [ ]:
# Reorder alerts
reorder_df = compute_reorder_alerts(store_inventory, orders_df)
print(f'Items needing reorder: {len(reorder_df)}')
if not reorder_df.empty:
    display(reorder_df[['sku_name', 'current_stock', 'dynamic_rop', 'static_rop',
                          'recommended_order_qty', 'avg_daily_demand']].head(15))

In [ ]:
# Full alert summary
summarize_alerts(expiry_df, reorder_df)

In [ ]:
# Expiry risk heatmap
if not at_risk.empty:
    fig, ax = plt.subplots(figsize=(10, max(3, len(at_risk) * 0.45)))
    color_map = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#2ecc71'}
    colors = [color_map[r] for r in at_risk['expiry_risk']]
    bars = ax.barh(at_risk['sku_name'], at_risk['sell_through_probability'], color=colors)
    ax.axvline(0.4, color='red', linestyle='--', alpha=0.7, label='HIGH threshold (0.4)')
    ax.axvline(0.7, color='orange', linestyle='--', alpha=0.7, label='MEDIUM threshold (0.7)')
    ax.set_xlabel('Sell-Through Probability')
    ax.set_title(f'Expiry Risk — Sell-Through Probability ({STORE_ID})', fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.show()
else:
    print('✅ No expiry risk items to plot')

---
## 6. V2 Ideas

| Priority | Feature | Notes |
|----------|---------|-------|
| 🔴 High | **Real-time pipeline** | Connect to live POS/WMS via Kafka or Flink |
| 🔴 High | **Dynamic markdown engine** | Auto-generate discount % for near-expiry items based on demand elasticity |
| 🟡 Med | **Per-store model tuning** | Each store may have very different demand patterns |
| 🟡 Med | **Anomaly detection** | Flag unusual demand spikes (promotions, viral moments) |
| 🟡 Med | **Supplier scoring** | Rank suppliers by lead time reliability and quality |
| 🟢 Nice | **Per-customer personalization** | Use order history to predict individual demand |
| 🟢 Nice | **Multi-store replenishment** | Balance stock across stores in real time |
| 🟢 Nice | **REST API** | Expose forecasts and alerts for warehouse management systems |

### Next steps for this notebook:
- Add backtesting: train on months 1–5, evaluate on month 6
- Compare Prophet vs LightGBM MAE/MAPE per category
- Add price elasticity curves for top-10 SKUs
- Build an interactive Streamlit/Gradio dashboard